# YOLO11n-seg + SimAM+CA + WIoU v3 Box Loss (Nhóm B — B1)

**Cải tiến:** Thay thế CIoU mặc định bằng WIoU v3 cho box regression loss.  
**Lý do:** Dataset annotation thủ công có noise — biên giới BG và WSSV không rõ ràng. WIoU v3 dùng dynamic non-monotonic focusing để giảm gradient từ outlier/noisy labels, tập trung học các mẫu 'mediocre' thay vì bị nhiễu bởi cả easy và hard outliers.  
**Kỳ vọng:** +2-4% mAP50-95 nhờ loại bỏ gradient độc hại từ nhãn sai.  
**Base model:** SimAM+CA (best model từ notebook combined).


## Portable Colab/Kaggle runtime paths

All generated data, cloned dependencies, training runs, and augmented-test outputs use the writable runtime directory selected below.


In [ ]:
from pathlib import Path
import os

if Path("/kaggle/working").exists():
    RUNTIME_ROOT = Path("/kaggle/working")
elif Path("/content").exists():
    RUNTIME_ROOT = Path("/content")
else:
    RUNTIME_ROOT = Path.cwd()

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(RUNTIME_ROOT)
print("Runtime root:", RUNTIME_ROOT)


In [ ]:
# Install dependencies
import importlib.util, subprocess, sys

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

print('Dependencies ready.')

In [ ]:
# ── GPU check ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Download dataset (Roboflow) ────────────────────────────────────────────
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ''
ROBOFLOW_WORKSPACE    = 'lets-try-this'
ROBOFLOW_PROJECT      = 'shrimpdishandsegv2'
ROBOFLOW_VERSION      = 1
ROBOFLOW_FORMAT       = 'yolo26'


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()


api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError('Missing Roboflow API key.')

rf      = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)

base_path = str(Path(dataset.location).resolve())
print('Dataset root:', base_path)

In [ ]:
# ── Grouped-stratified split (anti-leakage) ────────────────────────────────
import re, shutil, random
from collections import defaultdict, Counter
from pathlib import Path

SEED = 42
random.seed(SEED)
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
TRAIN_RATIO, VAL_RATIO = 0.80, 0.10

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


def parse_shrimp_group_key(image_name):
    stem  = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return f'unparsed::{stem}', 'unparsed', None, None
    disease  = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    return f'{disease.lower()}::{shrimp_id}', disease, shrimp_id, int(match.group('img_num'))


def image_files_in_split(split):
    d = Path(base_path) / split / 'images'
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def move_image_and_label(image_path, target_split):
    ti = Path(base_path) / target_split / 'images'
    tl = Path(base_path) / target_split / 'labels'
    ti.mkdir(parents=True, exist_ok=True)
    tl.mkdir(parents=True, exist_ok=True)
    lbl_src = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    img_dst = ti / image_path.name
    lbl_dst = tl / f'{image_path.stem}.txt'
    if image_path.resolve() != img_dst.resolve():
        shutil.move(str(image_path), str(img_dst))
    if lbl_src.exists():
        if lbl_src.resolve() != lbl_dst.resolve():
            shutil.move(str(lbl_src), str(lbl_dst))
    else:
        lbl_dst.write_text('')


def disease_for_group(filenames):
    diseases = [parse_shrimp_group_key(f)[1] for f in filenames]
    return Counter(diseases).most_common(1)[0][0]


def split_one_stratum(items):
    n = len(items)
    tr = int(TRAIN_RATIO * n)
    va = int(VAL_RATIO * n)
    te = n - tr - va
    if n >= 3:
        if va == 0: va, tr = 1, tr - 1
        if te == 0: te, tr = 1, tr - 1
    if tr < 1 and n > 0: tr = 1
    while tr + va + te > n: tr -= 1
    te = n - tr - va
    return items[:tr], items[tr:tr+va], items[tr+va:]


def remove_yolo_label_caches(root):
    for p in Path(root).glob('**/*.cache'): p.unlink()


# Rebuild pool from all splits then re-split
for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        os.makedirs(os.path.join(base_path, split, sub), exist_ok=True)

all_images = []
for split in ['train', 'valid', 'test']:
    all_images.extend(image_files_in_split(split))
for img in sorted(all_images):
    move_image_and_label(img, 'train')

groups = defaultdict(list)
for img in image_files_in_split('train'):
    gk, *_ = parse_shrimp_group_key(img.name)
    groups[gk].append(img.name)

strata = defaultdict(list)
for gk, filenames in sorted(groups.items()):
    strata[disease_for_group(filenames)].append((gk, filenames))

split_to_groups = {'train': [], 'valid': [], 'test': []}
rng = random.Random(SEED)
for disease, items in sorted(strata.items()):
    items = sorted(items, key=lambda x: x[0])
    rng.shuffle(items)
    tr, va, te = split_one_stratum(items)
    split_to_groups['train'].extend(tr)
    split_to_groups['valid'].extend(va)
    split_to_groups['test'].extend(te)
    print(f'  {disease}: {len(tr)} train, {len(va)} valid, {len(te)} test groups')

for split, sg in split_to_groups.items():
    for _, filenames in sg:
        for fn in filenames:
            move_image_and_label(Path(base_path) / 'train' / 'images' / fn, split)

# Leakage check
group_to_split = {}
for split in ['train', 'valid', 'test']:
    for ip in image_files_in_split(split):
        gk, *_ = parse_shrimp_group_key(ip.name)
        prev = group_to_split.setdefault(gk, split)
        if prev != split:
            raise RuntimeError(f'Leakage detected: {gk} in {prev} and {split}')
print('Shrimp-level leakage check PASSED.')
remove_yolo_label_caches(base_path)

for split in ['train', 'valid', 'test']:
    n = len(image_files_in_split(split))
    print(f'  {split}: {n} images')

In [ ]:
# ── Update data.yaml paths ─────────────────────────────────────────────────
import yaml

data_yaml_path = os.path.join(base_path, 'data.yaml')
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = f'{base_path}/train/images'
cfg['val']   = f'{base_path}/valid/images'
cfg['test']  = f'{base_path}/test/images'

with open(data_yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('data.yaml updated.')
print('Classes:', cfg.get('names'))

In [ ]:
# ── Patch WIoU v3 into ultralytics loss.py ─────────────────────────────────
import torch
import ultralytics.utils.loss as ult_loss
from ultralytics.utils.metrics import bbox_iou

OrigBboxLoss = ult_loss.BboxLoss


class WIoUv3BboxLoss(OrigBboxLoss):
    """Drop-in replacement: replaces CIoU with WIoU v3 non-monotonic focusing."""

    WIOU_SCALE = 1.0

    def __call__(self, *args, **kwargs):
        # Use *args so signature is forward-compatible with any ultralytics version.
        # Positional order (all versions): pred_dist, pred_bboxes, anchor_points,
        #   target_bboxes, target_scores, target_scores_sum, fg_mask, [extra...]
        loss_iou, loss_dfl = super().__call__(*args, **kwargs)

        pred_bboxes       = args[1]
        target_bboxes     = args[3]
        target_scores     = args[4]
        target_scores_sum = args[5]
        fg_mask           = args[6]

        if fg_mask.sum() > 0:
            with torch.no_grad():
                iou = bbox_iou(
                    pred_bboxes[fg_mask],
                    target_bboxes[fg_mask],
                    xywh=False, CIoU=True,
                ).squeeze(-1).clamp(0, 1)

            beta  = (iou - 0.5).abs()
            alpha = 1.0 / (2.0 * beta + 1e-7)
            alpha = alpha / alpha.mean().clamp(min=1e-7)

            iou_full = bbox_iou(
                pred_bboxes[fg_mask],
                target_bboxes[fg_mask],
                xywh=False, CIoU=True,
            ).squeeze(-1)

            w = (target_scores[fg_mask].max(-1).values if target_scores.ndim > 1
                 else target_scores[fg_mask])
            wiou_loss = (alpha * (1.0 - iou_full) * w).sum() / target_scores_sum
            loss_iou  = wiou_loss * self.WIOU_SCALE

        return loss_iou, loss_dfl


ult_loss.BboxLoss = WIoUv3BboxLoss
print('WIoU v3 BboxLoss patch applied.')

In [ ]:
# ── Patch attention modules into ultralytics namespace ─────────────────────
# CoordAtt args in YAML: [c1, reduction] where c1 = ACTUAL scaled channels
# from the FROM layer (64/128/256 for YOLO11n with width_multiple=0.25).
# ultralytics passes YAML args directly for custom modules — c1 must match.

import torch
import torch.nn as nn
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda   = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu  = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y     = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    # Channel-preserving: output channels == input channels (c1).
    # No separate c2 — ultralytics YAML args are (c1, reduction).
    def __init__(self, c1, reduction=32):
        super().__init__()
        mip = max(8, c1 // reduction)
        self.conv1  = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1    = nn.BatchNorm2d(mip)
        self.act    = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c1, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c1, 1, 1, 0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        return identity * a_h * a_w


for ns in [conv_module, nn_modules, tasks_module]:
    ns.SimAM    = SimAM
    ns.CoordAtt = CoordAtt
nn_modules.__all__ = list(set(list(getattr(nn_modules, '__all__', [])) + ['SimAM', 'CoordAtt']))
print('SimAM + CoordAtt registered in ultralytics namespace.')

In [ ]:
# ── Write SimAM+CA YAML to ultralytics cfg directory ───────────────────────
# CoordAtt args: [c1, reduction] where c1 = ACTUAL channels after width scaling.
# YOLO11n: width_multiple=0.25 → layer 16=64ch, layer 19=128ch, layer 22=256ch.
import ultralytics
from pathlib import Path

YAML_CONTENT = """
# YOLO11n-seg + SimAM + CA head — WIoU v3 experiment (Nhom B, B1)
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, CoordAtt, [64, 32]]
  - [23, 1, SimAM, []]

  - [19, 1, CoordAtt, [128, 32]]
  - [25, 1, SimAM, []]

  - [22, 1, CoordAtt, [256, 32]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]
""".strip()

ultralytics_root = Path(ultralytics.__file__).parent
yaml_dir  = ultralytics_root / 'cfg' / 'models' / '11'
yaml_dir.mkdir(parents=True, exist_ok=True)
yaml_path = yaml_dir / 'yolo11n-seg-simam-ca-wiouv3.yaml'
yaml_path.write_text(YAML_CONTENT)
print('YAML written to:', yaml_path)

In [ ]:
# ── Helper functions (reused from baseline) ────────────────────────────────
import gc, time, shutil
import yaml
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

EXPERIMENT_ROOT = RUNTIME_ROOT / 'shrimp_wiouv3'
RUNS_DIR        = EXPERIMENT_ROOT / 'runs' / 'segment'
REPORT_DIR      = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_PENALTY_WEIGHT      = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT    = 0.25
TRAIN_IMGSZ = 640


def find_image_for_label(image_dir, label_name):
    stem = Path(label_name).stem
    for ext in IMAGE_EXTENSIONS:
        c = Path(image_dir) / f'{stem}{ext}'
        if c.exists():
            return c
    return None


def write_data_yaml(dataset_dir, out_yaml):
    with open(data_yaml_path) as f:
        c = yaml.safe_load(f)
    c['train'] = str(Path(dataset_dir) / 'train' / 'images')
    c['val']   = str(Path(dataset_dir) / 'valid' / 'images')
    c['test']  = str(Path(dataset_dir) / 'test'  / 'images')
    with open(out_yaml, 'w') as f:
        yaml.safe_dump(c, f, sort_keys=False)


def copy_dataset_for_experiment(exp_key):
    dst = EXPERIMENT_ROOT / exp_key / 'dataset'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(base_path, dst, ignore=shutil.ignore_patterns('*.cache'))
    remove_yolo_label_caches(dst)
    return dst


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    si = Path(src_dataset) / split / 'images'
    sl = Path(src_dataset) / split / 'labels'
    di = Path(dst_dataset) / split / 'images'
    dl = Path(dst_dataset) / split / 'labels'
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    copied = 0
    for lp in sorted(sl.glob('*.txt')):
        lines = [l.strip() for l in lp.read_text().splitlines() if l.strip()]
        if bool(lines) != want_labeled:
            continue
        ip = find_image_for_label(si, lp.name)
        if ip is None:
            continue
        shutil.copy2(ip, di / ip.name)
        shutil.copy2(lp, dl / lp.name)
        copied += 1
    return copied


def make_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f'dataset_{state_name}_eval'
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ['images', 'labels']:
        (dst / 'train' / sub).mkdir(parents=True, exist_ok=True)
    for split in ['valid', 'test']:
        copy_split_by_label_state(src_dataset, dst, split, want_labeled)
    yp = dst / f'data_{state_name}.yaml'
    write_data_yaml(dst, yp)
    return dst, yp


def metric_value(metrics, dotted_path, default=float('nan')):
    obj = metrics
    for part in dotted_path.split('.'):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted(
        p for p in Path(images_dir).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        nan = float('nan')
        return dict(images=0, gt_total=0, pred_mask_total=0, mask_count_mae=nan,
                    disease_images=0, disease_box_miss_rate=nan, disease_mask_miss_rate=nan)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    mask_errs, disease_images, disease_box_miss = [], 0, 0
    gt_total = pred_total = 0
    for ip, res in zip(image_paths, results):
        lp = Path(labels_dir) / f'{ip.stem}.txt'
        gt = len([l for l in lp.read_text().splitlines() if l.strip()]) if lp.exists() else 0
        pm = len(res.masks) if res.masks is not None else 0
        pb = len(res.boxes) if res.boxes is not None else 0
        mask_errs.append(abs(pm - gt) / max(1, gt))
        if gt > 0:
            disease_images += 1
            disease_box_miss += int(pb == 0)
        gt_total += gt; pred_total += pm
    return dict(
        images=len(image_paths), gt_total=gt_total, pred_mask_total=pred_total,
        mask_count_mae=sum(mask_errs)/len(mask_errs) if mask_errs else float('nan'),
        disease_images=disease_images,
        disease_box_miss_rate=disease_box_miss/disease_images if disease_images else float('nan'),
        disease_mask_miss_rate=float('nan'),
    )


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = sorted(
        p for p in Path(images_dir).iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        nan = float('nan')
        return dict(healthy_images=0, healthy_mask_fp_rate=nan,
                    healthy_fp_masks_per_image=nan, healthy_avg_fp_confidence=0.0)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    with_fp = 0; mask_total = 0; confs = []
    for res in results:
        mc = len(res.masks) if res.masks is not None else 0
        if mc > 0:
            with_fp += 1
            mask_total += mc
            try:
                confs.extend(res.boxes.conf.cpu().tolist())
            except Exception:
                pass
    n = len(image_paths)
    return dict(
        healthy_images=n,
        healthy_mask_fp_rate=with_fp / n,
        healthy_fp_masks_per_image=mask_total / n,
        healthy_avg_fp_confidence=sum(confs)/len(confs) if confs else 0.0,
    )


def healthy_aware_score(labeled_map50, count_info, fp_info):
    return (labeled_map50
            - COUNT_PENALTY_WEIGHT * count_info['mask_count_mae']
            - DISEASE_MISS_PENALTY_WEIGHT * count_info['disease_box_miss_rate']
            - HEALTHY_FP_PENALTY_WEIGHT * fp_info['healthy_mask_fp_rate'])


def read_best_epoch_from_results(run_path):
    csv_path = Path(run_path) / 'results.csv'
    if not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    mc = 'metrics/mAP50(M)'
    if mc not in df.columns:
        return {'epochs_ran': len(df)}
    bi = df[mc].idxmax()
    best = df.iloc[bi]
    last = df.iloc[-1]
    return dict(
        epochs_ran=len(df),
        best_epoch_by_mask_map50=int(best.get('epoch', bi + 1)),
        best_val_mask_map50=float(best.get(mc, float('nan'))),
        last_train_seg_loss=float(last.get('train/seg_loss', float('nan'))),
        last_val_seg_loss=float(last.get('val/seg_loss', float('nan'))),
        seg_loss_gap_val_minus_train=float(last.get('val/seg_loss', 0) - last.get('train/seg_loss', 0)),
    )


def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as aug
        class _NoOp:
            contains_spatial = False
            def __init__(self, *a, **kw): self.transform = None
            def __call__(self, labels): return labels
        aug.Albumentations = _NoOp
    except Exception as e:
        print('Albumentations patch skipped:', e)


print('Helper functions defined.')

In [ ]:
# ── Training — SimAM+CA + WIoU v3 ─────────────────────────────────────────
CLEAN_TRAIN_ARGS = dict(
    auto_augment=None,
    erasing=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.01, hsv_s=0.35, hsv_v=0.20,
    degrees=0.0, translate=0.05, scale=0.20,
    shear=0.0, perspective=0.0, multi_scale=0.0, bgr=0.0,
)

exp_key  = 'simam_ca_wiouv3'
run_name = f'yolo11n-seg_{exp_key}'

dataset_dir = copy_dataset_for_experiment(exp_key)
exp_yaml    = dataset_dir / 'data.yaml'
write_data_yaml(dataset_dir, exp_yaml)

labeled_eval_dir, labeled_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'labeled_only', True)
healthy_eval_dir, healthy_eval_yaml = make_eval_dataset(dataset_dir, exp_key, 'healthy_only', False)

disable_ultralytics_albumentations()

# Build model from YAML and load pretrained backbone weights
yolo = YOLO(str(yaml_path))
try:
    yolo.load('yolo11n-seg.pt')
except Exception as e:
    print('Pretrained load warning:', e)

start = time.time()
yolo.train(
    data=str(exp_yaml),
    task='segment',
    imgsz=TRAIN_IMGSZ,
    epochs=100,
    batch=16,
    patience=30,
    seed=42,
    deterministic=True,
    workers=0,
    project=str(RUNS_DIR),
    name=run_name,
    exist_ok=True,
    pretrained=True,
    plots=True,
    verbose=True,
    **CLEAN_TRAIN_ARGS,
)
train_time_min = (time.time() - start) / 60
print(f'Training finished in {train_time_min:.1f} min.')

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
run_path   = RUNS_DIR / run_name
best_path  = run_path / 'weights' / 'best.pt'
best_model = YOLO(str(best_path))

full_val       = best_model.val(data=str(exp_yaml),          split='val',  imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
full_test      = best_model.val(data=str(exp_yaml),          split='test', imgsz=TRAIN_IMGSZ, plots=True,  verbose=False)
labeled_val    = best_model.val(data=str(labeled_eval_yaml), split='val',  imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
labeled_test   = best_model.val(data=str(labeled_eval_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)

labeled_test_count = count_prediction_errors(
    best_model, labeled_eval_dir/'test'/'images', labeled_eval_dir/'test'/'labels')
healthy_test_fp    = healthy_false_positive_summary(
    best_model, healthy_eval_dir/'test'/'images')

labeled_test_map50 = metric_value(labeled_test, 'seg.map50')
h_score = healthy_aware_score(labeled_test_map50, labeled_test_count, healthy_test_fp)

row = dict(
    experiment=exp_key,
    loss_variant='WIoU_v3',
    train_time_min=round(train_time_min, 2),
    full_val_mask_map50=metric_value(full_val,  'seg.map50'),
    full_test_mask_map50=metric_value(full_test, 'seg.map50'),
    labeled_val_mask_map50=metric_value(labeled_val,  'seg.map50'),
    labeled_test_mask_map50=labeled_test_map50,
    labeled_test_mask_map50_95=metric_value(labeled_test, 'seg.map'),
    healthy_test_mask_fp_rate=healthy_test_fp['healthy_mask_fp_rate'],
    healthy_aware_score=h_score,
    **labeled_test_count,
)
row.update(read_best_epoch_from_results(run_path))

results_df = pd.DataFrame([row])
results_df.to_csv(REPORT_DIR / 'wiouv3_results.csv', index=False)
display(results_df.T)

del best_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ── Visualise training curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_png = run_path / 'results.png'
if results_png.exists():
    plt.figure(figsize=(14, 10))
    plt.imshow(mpimg.imread(results_png))
    plt.axis('off')
    plt.title('WIoU v3 Training Curves')
    plt.show()
else:
    print('results.png not found at', results_png)

## Offline geometric augmentation of the test split — SimAM + CA with WIoU v3

Run this section only after leakage-safe splitting and model training. It creates an independent expanded test dataset from `test/images` and `test/labels` only. Each original image and its five transformed copies are evaluated separately; no TTA inverse mapping, voting, matching, or fusion is performed.


In [ ]:
# =========================================================
# Offline augmentation of TEST only — DPCA strong
# =========================================================
from pathlib import Path
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import yaml
from ultralytics import YOLO

TEST_AUG_CONFIG = {
    "enabled": True,
    "imgsz": 640,
    "transforms": [
        "original",
        "horizontal_flip",
        "vertical_flip",
        "rotate_90",
        "rotate_180",
        "rotate_270",
    ],
    "conf": 0.25,
    "iou_nms": 0.70,
    "max_det": 300,
    "agnostic_nms": False,
}


def apply_test_transform(image, transform_name):
    # OpenCV geometric operations require a supported, contiguous array.
    if image.dtype == np.bool_:
        image = image.astype(np.uint8) * 255
    elif image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    image = np.ascontiguousarray(image)

    if transform_name == "original":
        return image.copy()
    if transform_name == "horizontal_flip":
        return cv2.flip(image, 1)
    if transform_name == "vertical_flip":
        return cv2.flip(image, 0)
    if transform_name == "rotate_90":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if transform_name == "rotate_180":
        return cv2.rotate(image, cv2.ROTATE_180)
    if transform_name == "rotate_270":
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError(f"Unknown transform: {transform_name}")


def transform_normalized_polygon(points, transform_name):
    """Transform normalized YOLO segmentation coordinates with the image."""
    transformed = points.copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    if transform_name == "original":
        pass
    elif transform_name == "horizontal_flip":
        transformed[:, 0], transformed[:, 1] = 1.0 - x, y
    elif transform_name == "vertical_flip":
        transformed[:, 0], transformed[:, 1] = x, 1.0 - y
    elif transform_name == "rotate_90":
        transformed[:, 0], transformed[:, 1] = 1.0 - y, x
    elif transform_name == "rotate_180":
        transformed[:, 0], transformed[:, 1] = 1.0 - x, 1.0 - y
    elif transform_name == "rotate_270":
        transformed[:, 0], transformed[:, 1] = y, 1.0 - x
    else:
        raise ValueError(f"Unknown transform: {transform_name}")
    return transformed.clip(0.0, 1.0)


def transform_segmentation_label(source_label, destination_label, transform_name):
    output_lines = []
    if source_label.exists():
        for line in source_label.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) < 7 or (len(parts) - 1) % 2:
                raise ValueError(f"Invalid segmentation polygon in {source_label}: {line}")
            class_id = parts[0]
            points = np.asarray([float(value) for value in parts[1:]], dtype=np.float64).reshape(-1, 2)
            points = transform_normalized_polygon(points, transform_name)
            coordinates = " ".join(f"{value:.8f}" for value in points.reshape(-1))
            output_lines.append(f"{class_id} {coordinates}")
    destination_label.write_text("\n".join(output_lines) + ("\n" if output_lines else ""))


def build_augmented_test_dataset(source_dataset, output_dataset, config):
    source_dataset = Path(source_dataset)
    output_dataset = Path(output_dataset)
    source_images = source_dataset / "test" / "images"
    source_labels = source_dataset / "test" / "labels"
    output_images = output_dataset / "test" / "images"
    output_labels = output_dataset / "test" / "labels"

    # This directory is derived only from the already-isolated test split.
    if output_dataset.exists():
        shutil.rmtree(output_dataset)
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    image_paths = []
    for extension in IMAGE_EXTENSIONS:
        image_paths.extend(source_images.glob(f"*{extension}"))
    image_paths = sorted(set(image_paths))

    rows = []
    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(image_path)
        source_label = source_labels / f"{image_path.stem}.txt"
        for transform_name in config["transforms"]:
            output_name = f"{transform_name}__{image_path.stem}.png"
            output_image = output_images / output_name
            output_label = output_labels / f"{Path(output_name).stem}.txt"
            transformed_image = apply_test_transform(image, transform_name)
            if not cv2.imwrite(str(output_image), transformed_image):
                raise RuntimeError(f"Could not write {output_image}")
            transform_segmentation_label(source_label, output_label, transform_name)
            rows.append({
                "source_image": image_path.name,
                "augmented_image": output_name,
                "transform": transform_name,
                "label_nonempty": bool(output_label.read_text().strip()),
            })

    expected = len(image_paths) * len(config["transforms"])
    if len(rows) != expected:
        raise RuntimeError(f"Expected {expected} augmented samples, created {len(rows)}")
    return image_paths, pd.DataFrame(rows)



MODULE_SLUG = 'simam_ca_wiouv3'
tta_checkpoint = Path(best_path)
source_test_dataset = Path(dataset_dir)
if not tta_checkpoint.exists():
    raise FileNotFoundError(f"Missing trained checkpoint: {tta_checkpoint}")
if not (source_test_dataset / "test" / "images").exists():
    raise FileNotFoundError(f"Missing source test split: {source_test_dataset}")

runtime_root = RUNTIME_ROOT
TEST_AUG_ROOT = runtime_root / "test_augmentation" / 'dpca_strong'
augmented_dataset = TEST_AUG_ROOT / "dataset"
source_test_images, augmentation_manifest = build_augmented_test_dataset(
    source_test_dataset, augmented_dataset, TEST_AUG_CONFIG
)

# Reuse class metadata from the experiment YAML, changing only the TEST path.
source_yaml = source_test_dataset / "data.yaml"
with open(source_yaml, "r") as stream:
    augmented_yaml_content = yaml.safe_load(stream)
augmented_yaml_content["test"] = str((augmented_dataset / "test" / "images").resolve())
augmented_yaml = augmented_dataset / "data_augmented_test.yaml"
with open(augmented_yaml, "w") as stream:
    yaml.safe_dump(augmented_yaml_content, stream, sort_keys=False)

augmentation_manifest.to_csv(TEST_AUG_ROOT / "augmentation_manifest.csv", index=False)
(TEST_AUG_ROOT / "test_augmentation_config.json").write_text(json.dumps(TEST_AUG_CONFIG, indent=2))

print(f"Original test images: {len(source_test_images)}")
print(f"Transforms per image: {len(TEST_AUG_CONFIG['transforms'])}")
print(f"Expanded test images: {len(augmentation_manifest)}")
print("Augmented test YAML:", augmented_yaml)

# Evaluate only after the model has finished training; augmented samples never enter train/val.
test_aug_model = YOLO(str(tta_checkpoint))
augmented_test_metrics = test_aug_model.val(
    data=str(augmented_yaml),
    split="test",
    imgsz=TEST_AUG_CONFIG["imgsz"],
    conf=TEST_AUG_CONFIG["conf"],
    iou=TEST_AUG_CONFIG["iou_nms"],
    max_det=TEST_AUG_CONFIG["max_det"],
    agnostic_nms=TEST_AUG_CONFIG["agnostic_nms"],
    plots=True,
    verbose=False,
)

augmented_test_result = {
    "original_test_images": len(source_test_images),
    "transforms_per_image": len(TEST_AUG_CONFIG["transforms"]),
    "expanded_test_images": len(augmentation_manifest),
    "box_precision": float(augmented_test_metrics.box.mp),
    "box_recall": float(augmented_test_metrics.box.mr),
    "box_map50": float(augmented_test_metrics.box.map50),
    "box_map50_95": float(augmented_test_metrics.box.map),
    "mask_precision": float(augmented_test_metrics.seg.mp),
    "mask_recall": float(augmented_test_metrics.seg.mr),
    "mask_map50": float(augmented_test_metrics.seg.map50),
    "mask_map50_95": float(augmented_test_metrics.seg.map),
}
pd.DataFrame([augmented_test_result]).to_csv(TEST_AUG_ROOT / "augmented_test_metrics.csv", index=False)
(TEST_AUG_ROOT / "augmented_test_metrics.json").write_text(json.dumps(augmented_test_result, indent=2))
display(pd.DataFrame([augmented_test_result]))


# Display predictions from the expanded test set, following the original notebook style.
# Select up to two samples per transform so every corruption is represented.
import matplotlib.pyplot as plt

preview_rows = (
    augmentation_manifest.sort_values(["transform", "augmented_image"])
    .groupby("transform", sort=False, group_keys=False)
    .head(2)
)
preview_paths = [
    augmented_dataset / "test" / "images" / image_name
    for image_name in preview_rows["augmented_image"].tolist()
]

if preview_paths:
    preview_results = test_aug_model.predict(
        source=[str(path) for path in preview_paths],
        imgsz=TEST_AUG_CONFIG["imgsz"],
        conf=TEST_AUG_CONFIG["conf"],
        iou=TEST_AUG_CONFIG["iou_nms"],
        max_det=TEST_AUG_CONFIG["max_det"],
        agnostic_nms=TEST_AUG_CONFIG["agnostic_nms"],
        save=False,
        verbose=False,
    )
    preview_transform_by_name = dict(
        zip(preview_rows["augmented_image"], preview_rows["transform"])
    )
    for image_path, result in zip(preview_paths, preview_results):
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        plotted_bgr = result.plot()
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB))
        plt.title(
            f"{preview_transform_by_name[image_path.name]} | {image_path.name} | "
            f"boxes={box_count}, masks={mask_count}"
        )
        plt.axis("off")
        plt.show()
else:
    print("No augmented test images were available for prediction preview.")


print("Offline augmented-test evaluation complete:", TEST_AUG_ROOT)
